
#  Semana 02 — Ejercicios de Optimización con PuLP

## Cuaderno de trabajo para estudiantes

Este notebook contiene **únicamente los planteamientos de los ejercicios**.  
El objetivo es que cada estudiante formule y programe su propia solución utilizando **PuLP en Python**.

---

##  Indicaciones generales

Para cada ejercicio se recomienda seguir esta secuencia:

1. Identificar las **variables de decisión**.
2. Determinar si son **continuas, enteras o binarias**.
3. Formular la **función objetivo**.
4. Escribir matemáticamente las **restricciones**.
5. Implementar el modelo en **PuLP**.
6. Resolver el modelo.
7. Revisar el estado de la solución.
8. Validar manualmente las restricciones.
9. Interpretar el resultado en el contexto del problema.

> 💡 No basta con obtener números. Debe justificarse por qué la solución encontrada es factible y qué significa en el contexto del problema.



##  Preparación del entorno

Utilice la siguiente celda únicamente para importar PuLP.

Si la librería no está instalada en su entorno, instálela antes de continuar.


In [ ]:

# Importar PuLP
import pulp



#  Ejercicio 1 — Dimensionamiento de infraestructura Cloud

##  Planteamiento

Una empresa debe contratar instancias de tres tipos para soportar una nueva plataforma.  
Se desea cubrir una capacidad mínima de **CPU** y **memoria RAM** al menor costo mensual posible.

### 📊 Datos

| Tipo | Costo mensual | vCPU | RAM |
|---|---:|---:|---:|
| A — Standard | $120 | 8 | 32 GB |
| B — Compute | $180 | 16 | 64 GB |
| C — High Capacity | $260 | 32 | 96 GB |

###  Condiciones

- Se requieren al menos **160 vCPU**.
- Se requieren al menos **520 GB de RAM**.
- Por resiliencia, deben contratarse al menos **3 instancias tipo C**.
- No pueden administrarse más de **15 instancias en total**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe identificar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricciones;
- solución óptima;
- costo mínimo;
- validación de CPU, RAM y número total de instancias;
- interpretación de la solución.


In [11]:

# EJERCICIO 1

import pulp

costos = {"A": 120, "B": 180, "C": 260}
cpu = {"A": 8, "B": 16, "C": 32}
ram = {"A": 32, "B": 64, "C": 96}
cpu_minima = 160
ram_minima = 520
instancia_c_minimo = 3
instancias_maximas = 15

modelo = pulp.LpProblem("Dimensionamiento_Cloud", pulp.LpMinimize)

x = {
    tipo: pulp.LpVariable(f"Instancias_{tipo}", lowBound=0, cat="Integer")
    for tipo in costos
}


# FUNCIÓN OBJETIVO

modelo += pulp.lpSum(costos[tipo] * x[tipo] for tipo in costos), "Costo_total"


# RESTRICCIONES

# La cantidad total de vCPU debe ser al menos 160
modelo += pulp.lpSum(cpu[tipo] * x[tipo] for tipo in costos) >= cpu_minima, "CPU_minima"

# La cantidad total de RAM debe ser al menos 520 GB
modelo += pulp.lpSum(ram[tipo] * x[tipo] for tipo in costos) >= ram_minima, "RAM_minima"

# Debemos contratar al menos 3 instancias tipo C
modelo += x["C"] >= instancia_c_minimo, "Minimo_instancias_C"

# No podemos contratar más de 15 instancias en total
modelo += pulp.lpSum(x[tipo] for tipo in costos) <= instancias_maximas, "Maximo_instancias"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

# Mostramos cuántas instancias de cada tipo debemos contratar
for tipo in costos:
    print(f"Instancias {tipo}: {x[tipo].varValue}")

# Mostramos el costo mínimo encontrado
print("Costo mínimo ($):", pulp.value(modelo.objective))


Estado: Optimal
Instancias A: 0.0
Instancias B: 1.0
Instancias C: 5.0
Costo mínimo ($): 1480.0



##  Reto de ampliación

Modifique el modelo anterior considerando ahora:

- demanda mínima de **200 vCPU**;
- demanda mínima de **640 GB de RAM**;
- obligación de contratar al menos **2 instancias tipo A** por compatibilidad con software legado.

Compare el nuevo costo con el modelo original.


In [23]:
# RETO EJERCICIO 1
import pulp

# Crear el problema
modelo = pulp.LpProblem("Infraestructura_Cloud_Reto", pulp.LpMinimize)

# Variables de decisión
A = pulp.LpVariable("Instancias_A", lowBound=0, cat="Integer")
B = pulp.LpVariable("Instancias_B", lowBound=0, cat="Integer")
C = pulp.LpVariable("Instancias_C", lowBound=0, cat="Integer")

# Función objetivo
modelo += 120*A + 180*B + 260*C

# Restricciones
modelo += 8*A + 16*B + 32*C >= 200
modelo += 32*A + 64*B + 96*C >= 640
modelo += C >= 3
modelo += A + B + C <= 15
modelo += A >= 2

# Resolver
modelo.solve()

# Resultados
print("Estado:", pulp.LpStatus[modelo.status])
print("A:", A.value())
print("B:", B.value())
print("C:", C.value())

print("Costo mínimo:", pulp.value(modelo.objective))

# Validaciones
cpu = 8*A.value() + 16*B.value() + 32*C.value()
ram = 32*A.value() + 64*B.value() + 96*C.value()
total = A.value() + B.value() + C.value()

print("CPU:", cpu, "vCPU")
print("RAM:", ram, "GB")
print("Total de instancias:", total)

Estado: Optimal
A: 2.0
B: 0.0
C: 6.0
Costo mínimo: 1800.0
CPU: 208.0 vCPU
RAM: 640.0 GB
Total de instancias: 8.0



#  Ejercicio 2 — Enrutamiento de tráfico entre enlaces WAN

##  Planteamiento

Un centro de datos debe distribuir **1,000 Mbps** entre tres enlaces WAN.  
Los enlaces tienen distintos costos, capacidades y latencias.

Se desea **minimizar el costo del tráfico**, pero la latencia promedio ponderada no debe superar **40 ms**.

###  Datos

| Enlace | Costo por Mbps | Capacidad máxima | Latencia |
|---|---:|---:|---:|
| L1 | $0.08 | 400 Mbps | 20 ms |
| L2 | $0.05 | 500 Mbps | 35 ms |
| L3 | $0.03 | 600 Mbps | 60 ms |

###  Condiciones

- Todo el tráfico debe ser enviado.
- El tráfico puede fraccionarse entre los enlaces.
- No debe superarse la capacidad máxima de cada enlace.
- La latencia promedio ponderada debe ser como máximo **40 ms**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe determinar:

- variables de decisión;
- tipo de variables;
- función objetivo;
- restricción de balance;
- restricciones de capacidad;
- restricción de latencia promedio;
- costo mínimo;
- distribución óptima de tráfico;
- latencia promedio resultante.


In [34]:
# EJERCICIO 2
# Escriba aquí su modelo en PuLP.
import pulp

costo_enlaces = {"L1": 0.08, "L2": 0.05, "L3": 0.03}
capacidad_enlaces = {"L1": 400, "L2": 500, "L3": 600}
latencia_enlaces = {"L1": 20, "L2": 35, "L3": 60}
latencia_promedio_maxima = 40
trafico_total = 1000

modelo = pulp.LpProblem("Enrutamiento_Trafico_WAN", pulp.LpMinimize)

x = {
    enlace: pulp.LpVariable(f"Trafico_{enlace}", lowBound=0, cat="Continuous")
    for enlace in costo_enlaces
}


# FUNCIÓN OBJETIVO

modelo += pulp.lpSum(costo_enlaces[i] * x[i] for i in costo_enlaces), "Costo_total"


# RESTRICCIONES

# Todo el tráfico debe ser enviado
modelo += pulp.lpSum(x[i] for i in costo_enlaces) == trafico_total, "Balance_trafico"

# No superar la capacidad máxima de cada enlace
for i in costo_enlaces:
    modelo += x[i] <= capacidad_enlaces[i], f"Capacidad_{i}"

# La latencia promedio ponderada no debe superar 40 ms
modelo += pulp.lpSum(latencia_enlaces[i] * x[i] for i in costo_enlaces) <= latencia_promedio_maxima * trafico_total, "Latencia_promedio"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

for i in costo_enlaces:
    print(f"Tráfico {i}: {x[i].varValue} Mbps")

latencia_resultante = sum(latencia_enlaces[i] * x[i].varValue for i in costo_enlaces) / trafico_total
print("Latencia promedio resultante (ms):", latencia_resultante)
print("Costo mínimo ($):", pulp.value(modelo.objective))



Estado: Optimal
Tráfico L1: 187.5 Mbps
Tráfico L2: 500.0 Mbps
Tráfico L3: 312.5 Mbps
Latencia promedio resultante (ms): 40.0
Costo mínimo ($): 49.375



##  Reto de ampliación

1. Reduzca la latencia máxima permitida a **35 ms**.
2. Compare el nuevo costo con el problema original.
3. Luego simule una caída parcial de L2 reduciendo su capacidad a **200 Mbps**.
4. Determine si el problema sigue siendo factible.


In [44]:
# RETO EJERCICIO 2
import pulp

costo_enlaces = {"L1": 0.08, "L2": 0.05, "L3": 0.03}
capacidad_enlaces = {"L1": 400, "L2": 200, "L3": 600}  # L2 con caída de capacidad
latencia_enlaces = {"L1": 20, "L2": 35, "L3": 60}
latencia_promedio_maxima = 35  # reducida según el reto
trafico_total = 1000

modelo = pulp.LpProblem("Enrutamiento_Trafico_WAN_Reto", pulp.LpMinimize)

x = {
    enlace: pulp.LpVariable(f"Trafico_{enlace}", lowBound=0, cat="Continuous")
    for enlace in costo_enlaces
}


# FUNCIÓN OBJETIVO

modelo += pulp.lpSum(costo_enlaces[i] * x[i] for i in costo_enlaces), "Costo_total"


# RESTRICCIONES

# Todo el tráfico debe ser enviado
modelo += pulp.lpSum(x[i] for i in costo_enlaces) == trafico_total, "Balance_trafico"

# No superar la capacidad máxima de cada enlace (L2 con caída a 200 Mbps)
for i in costo_enlaces:
    modelo += x[i] <= capacidad_enlaces[i], f"Capacidad_{i}"

# La latencia promedio ponderada no debe superar 35 ms
modelo += pulp.lpSum(latencia_enlaces[i] * x[i] for i in costo_enlaces) <= latencia_promedio_maxima * trafico_total, "Latencia_promedio"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

if modelo.status == pulp.LpStatusOptimal:
    for i in costo_enlaces:
        print(f"Tráfico {i}: {x[i].varValue} Mbps")

    latencia_resultante = sum(latencia_enlaces[i] * x[i].varValue for i in costo_enlaces) / trafico_total
    print("Latencia promedio resultante (ms):", latencia_resultante)
    print("Costo mínimo ($):", pulp.value(modelo.objective))
else:
    print("El problema no es factible con estas condiciones.")

Estado: Infeasible
El problema no es factible con estas condiciones.



#  Ejercicio 3 — Portafolio de controles de ciberseguridad

##  Planteamiento

El CISO dispone de un presupuesto limitado y debe seleccionar controles de seguridad.  
Cada control tiene un costo y una puntuación estimada de reducción de riesgo.

### 📊 Datos

| Control | Costo | Reducción de riesgo |
|---|---:|---:|
| MFA | 12 | 25 |
| EDR | 20 | 30 |
| SIEM | 25 | 28 |
| PAM | 18 | 24 |
| Backup inmutable | 15 | 22 |
| Capacitación | 8 | 12 |

###  Condiciones

- El presupuesto máximo es **70**.
- SIEM solo puede implementarse si también se selecciona EDR.
- PAM requiere que MFA esté seleccionado.
- Debe elegirse al menos una medida entre **Backup inmutable** y **Capacitación**.
- Deben seleccionarse al menos **4 controles**.

---

## Trabajo del estudiante

Construya un modelo de programación binaria que **maximice la reducción total de riesgo**.

Debe incluir:

- una variable binaria por control;
- función objetivo;
- restricción presupuestaria;
- restricciones de dependencia;
- restricción de continuidad;
- número mínimo de controles;
- interpretación de los controles seleccionados.


In [53]:
# EJERCICIO 3
# Escriba aquí su modelo en PuLP.

import pulp

costo_control = {
    "MFA": 12,
    "EDR": 20,
    "SIEM": 25,
    "PAM": 18,
    "Backup": 15,
    "Capacitacion": 8,
}
reduccion_riesgo = {
    "MFA": 25,
    "EDR": 30,
    "SIEM": 28,
    "PAM": 24,
    "Backup": 22,
    "Capacitacion": 12,
}
presupuesto_maximo = 70
controles_minimos = 4

modelo = pulp.LpProblem("Portafolio_Ciberseguridad", pulp.LpMaximize)

x = {
    control: pulp.LpVariable(f"Selecciona_{control}", cat="Binary")
    for control in costo_control
}


# FUNCIÓN OBJETIVO

modelo += pulp.lpSum(reduccion_riesgo[c] * x[c] for c in costo_control), "Riesgo_reducido_total"


# RESTRICCIONES

# No superar el presupuesto disponible
modelo += pulp.lpSum(costo_control[c] * x[c] for c in costo_control) <= presupuesto_maximo, "Presupuesto"

# SIEM solo puede implementarse si también se selecciona EDR
modelo += x["SIEM"] <= x["EDR"], "Dependencia_SIEM_EDR"

# PAM requiere que MFA esté seleccionado
modelo += x["PAM"] <= x["MFA"], "Dependencia_PAM_MFA"

# Debe elegirse al menos una medida entre Backup inmutable y Capacitación
modelo += x["Backup"] + x["Capacitacion"] >= 1, "Continuidad_minima"

# Deben seleccionarse al menos 4 controles
modelo += pulp.lpSum(x[c] for c in costo_control) >= controles_minimos, "Controles_minimos"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

for c in costo_control:
    if x[c].varValue == 1:
        print(f"Control seleccionado: {c}")

print("Reducción de riesgo total:", pulp.value(modelo.objective))


Estado: Optimal
Control seleccionado: MFA
Control seleccionado: EDR
Control seleccionado: PAM
Control seleccionado: Backup
Reducción de riesgo total: 101.0



##  Reto de ampliación

Agregue las siguientes reglas:

1. SIEM y una herramienta *legacy* no pueden coexistir.
2. Si se elige **Backup inmutable**, también debe elegirse **MFA**.

Formule las desigualdades binarias correspondientes.

> Nota: si desea calcular un nuevo óptimo incluyendo una herramienta *legacy*, deberá definir también su costo y su contribución a la reducción de riesgo.


In [61]:
# RETO EJERCICIO 3
import pulp

costo_control = {
    "MFA": 12,
    "EDR": 20,
    "SIEM": 25,
    "PAM": 18,
    "Backup": 15,
    "Capacitacion": 8,
    "Legacy": 10,
}
reduccion_riesgo = {
    "MFA": 25,
    "EDR": 30,
    "SIEM": 28,
    "PAM": 24,
    "Backup": 22,
    "Capacitacion": 12,
    "Legacy": 15,
}
presupuesto_maximo = 70
controles_minimos = 4

modelo = pulp.LpProblem("Portafolio_Ciberseguridad_Reto", pulp.LpMaximize)

x = {
    control: pulp.LpVariable(f"Selecciona_{control}", cat="Binary")
    for control in costo_control
}


# FUNCIÓN OBJETIVO

modelo += pulp.lpSum(reduccion_riesgo[c] * x[c] for c in costo_control), "Riesgo_reducido_total"


# RESTRICCIONES

# No superar el presupuesto disponible
modelo += pulp.lpSum(costo_control[c] * x[c] for c in costo_control) <= presupuesto_maximo, "Presupuesto"

# SIEM solo puede implementarse si también se selecciona EDR
modelo += x["SIEM"] <= x["EDR"], "Dependencia_SIEM_EDR"

# PAM requiere que MFA esté seleccionado
modelo += x["PAM"] <= x["MFA"], "Dependencia_PAM_MFA"

# Debe elegirse al menos una medida entre Backup inmutable y Capacitación
modelo += x["Backup"] + x["Capacitacion"] >= 1, "Continuidad_minima"

# Deben seleccionarse al menos 4 controles
modelo += pulp.lpSum(x[c] for c in costo_control) >= controles_minimos, "Controles_minimos"

# SIEM y la herramienta legacy no pueden coexistir
modelo += x["SIEM"] + x["Legacy"] <= 1, "Exclusion_SIEM_Legacy"

# Si se elige Backup inmutable, también debe elegirse MFA
modelo += x["Backup"] <= x["MFA"], "Dependencia_Backup_MFA"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

for c in costo_control:
    if x[c].varValue == 1:
        print(f"Control seleccionado: {c}")

print("Reducción de riesgo total:", pulp.value(modelo.objective))

Estado: Optimal
Control seleccionado: MFA
Control seleccionado: EDR
Control seleccionado: PAM
Control seleccionado: Capacitacion
Control seleccionado: Legacy
Reducción de riesgo total: 106.0



#  Ejercicio 4 — Distribución de respaldos entre niveles de almacenamiento

##  Planteamiento

Una organización debe almacenar **80 TB** de respaldos utilizando tres niveles: Hot, Warm y Cold.

Se desea **minimizar el costo mensual**, manteniendo una disponibilidad mínima y un tiempo promedio de recuperación aceptable.

###  Datos

| Nivel | Costo por TB | Tiempo de recuperación |
|---|---:|---:|
| Hot | $18 | 0.5 h |
| Warm | $10 | 4 h |
| Cold | $4 | 12 h |

###  Condiciones

- El total almacenado debe ser exactamente **80 TB**.
- Al menos **15 TB** deben permanecer en Hot.
- Al menos **20 TB** deben permanecer en Warm.
- Cold no puede superar **45 TB**.
- El tiempo promedio ponderado de recuperación debe ser como máximo **8 horas**.

---

##  Trabajo del estudiante

Formule y resuelva el modelo en PuLP.

Debe calcular:

- cantidad óptima de TB en cada nivel;
- costo mensual mínimo;
- tiempo promedio de recuperación;
- cumplimiento de todas las restricciones.


In [91]:
# EJERCICIO 4
# Escriba aquí su modelo en PuLP.
import pulp

# Crear el problema
modelo = pulp.LpProblem("Distribucion_Backups", pulp.LpMinimize)

# Variables de decisión
hot = pulp.LpVariable("Hot", lowBound=0, cat="Continuous")
warm = pulp.LpVariable("Warm", lowBound=0, cat="Continuous")
cold = pulp.LpVariable("Cold", lowBound=0, cat="Continuous")

# Función objetivo
modelo += (
    18*hot +
    10*warm +
    4*cold
), "Costo_Mensual"

# Restricción: total exactamente 80 TB
modelo += hot + warm + cold == 80, "Almacenamiento_Total"

# Restricción: mínimo Hot
modelo += hot >= 15, "Minimo_Hot"

# Restricción: mínimo Warm
modelo += warm >= 20, "Minimo_Warm"

# Restricción: máximo Cold
modelo += cold <= 45, "Maximo_Cold"

# Restricción: tiempo promedio de recuperación
modelo += (
    0.5*hot +
    4*warm +
    12*cold
) <= 640, "Tiempo_Recuperacion"

# Resolver
modelo.solve()

# Mostrar resultados
print("Estado:", pulp.LpStatus[modelo.status])

print("Hot:", hot.value(), "TB")
print("Warm:", warm.value(), "TB")
print("Cold:", cold.value(), "TB")

print("Costo mínimo mensual: $", pulp.value(modelo.objective))

# Validación del almacenamiento
total_tb = hot.value() + warm.value() + cold.value()

print("Almacenamiento total:", total_tb, "TB")

# Validación del tiempo promedio
tiempo_total = (
    0.5*hot.value() +
    4*warm.value() +
    12*cold.value()
)

tiempo_promedio = tiempo_total / 80

print("Tiempo promedio de recuperación:",
      tiempo_promedio, "horas")

Estado: Optimal
Hot: 15.0 TB
Warm: 20.0 TB
Cold: 45.0 TB
Costo mínimo mensual: $ 650.0
Almacenamiento total: 80.0 TB
Tiempo promedio de recuperación: 7.84375 horas



##  Reto de ampliación

1. Elimine la restricción que limita Cold a **45 TB**.
2. Observe si la restricción de RTO se vuelve determinante.
3. Luego exija un RTO promedio máximo de **6 horas**.
4. Compare la nueva distribución y el costo.


In [97]:
# RETO EJERCICIO 4
import pulp

costo_nivel = {"Hot": 18, "Warm": 10, "Cold": 4}
tiempo_recuperacion = {"Hot": 0.5, "Warm": 4, "Cold": 12}
total_respaldos = 80
minimo_hot = 15
minimo_warm = 20
tiempo_promedio_maximo = 6  # RTO exigido en el reto

modelo = pulp.LpProblem("Distribucion_Respaldos_Reto", pulp.LpMinimize)

x = {
    nivel: pulp.LpVariable(f"TB_{nivel}", lowBound=0, cat="Continuous")
    for nivel in costo_nivel
}


# FUNCIÓN OBJETIVO

modelo += pulp.lpSum(costo_nivel[n] * x[n] for n in costo_nivel), "Costo_total"


# RESTRICCIONES

# El total almacenado debe ser exactamente 80 TB
modelo += pulp.lpSum(x[n] for n in costo_nivel) == total_respaldos, "Total_respaldos"

# Al menos 15 TB deben permanecer en Hot
modelo += x["Hot"] >= minimo_hot, "Minimo_Hot"

# Al menos 20 TB deben permanecer en Warm
modelo += x["Warm"] >= minimo_warm, "Minimo_Warm"

# (Se elimina el límite de 45 TB en Cold, según el reto)

# El tiempo promedio ponderado de recuperación debe ser como máximo 6 horas
modelo += pulp.lpSum(tiempo_recuperacion[n] * x[n] for n in costo_nivel) <= tiempo_promedio_maximo * total_respaldos, "Tiempo_promedio"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

for n in costo_nivel:
    print(f"TB en {n}: {x[n].varValue}")

tiempo_resultante = sum(tiempo_recuperacion[n] * x[n].varValue for n in costo_nivel) / total_respaldos
print("Tiempo promedio de recuperación (h):", tiempo_resultante)
print("Costo mínimo ($):", pulp.value(modelo.objective))

Estado: Optimal
TB en Hot: 15.0
TB en Warm: 38.4375
TB en Cold: 26.5625
Tiempo promedio de recuperación (h): 6.0
Costo mínimo ($): 760.625



#  Ejercicio 5 — Localización de nodos Edge y asignación de regiones

##  Planteamiento

Una compañía debe decidir qué nodos Edge abrir y a qué nodo asignar cada región de usuarios.

Abrir un nodo genera un **costo fijo**.  
Atender una región desde un nodo genera un costo asociado con distancia, latencia y tráfico.

###  Nodos disponibles

| Nodo | Capacidad | Costo fijo |
|---|---:|---:|
| N1 | 80 | 100 |
| N2 | 70 | 90 |
| N3 | 75 | 95 |

### Demandas regionales

| Región | Demanda |
|---|---:|
| R1 | 40 |
| R2 | 35 |
| R3 | 30 |
| R4 | 25 |

### Costos unitarios por región y nodo

| Región | N1 | N2 | N3 |
|---|---:|---:|---:|
| R1 | 2 | 5 | 7 |
| R2 | 4 | 2 | 6 |
| R3 | 6 | 3 | 2 |
| R4 | 7 | 5 | 2 |

###  Condiciones

- Cada región debe asignarse exactamente a **un nodo**.
- Una región solo puede asignarse a un nodo que haya sido abierto.
- La suma de las demandas asignadas a cada nodo no puede superar su capacidad.
- Las decisiones de apertura y asignación son binarias.

---

##  Trabajo del estudiante

Construya un modelo que minimice:

- costos fijos de apertura;
- más costos de servicio de las regiones.

Debe determinar:

- nodos que deben abrirse;
- asignación de cada región;
- costo total;
- utilización de capacidad por nodo.


In [102]:
# EJERCICIO 5
# Escriba aquí su modelo en PuLP.
import pulp

nodos = ["N1", "N2", "N3"]
regiones = ["R1", "R2", "R3", "R4"]

capacidad_nodo = {"N1": 80, "N2": 70, "N3": 75}
costo_fijo_nodo = {"N1": 100, "N2": 90, "N3": 95}
demanda_region = {"R1": 40, "R2": 35, "R3": 30, "R4": 25}

costo_servicio = {
    "R1": {"N1": 2, "N2": 5, "N3": 7},
    "R2": {"N1": 4, "N2": 2, "N3": 6},
    "R3": {"N1": 6, "N2": 3, "N3": 2},
    "R4": {"N1": 7, "N2": 5, "N3": 2},
}

modelo = pulp.LpProblem("Localizacion_Nodos_Edge", pulp.LpMinimize)

y = {nodo: pulp.LpVariable(f"Abre_{nodo}", cat="Binary") for nodo in nodos}
z = {
    (region, nodo): pulp.LpVariable(f"Asigna_{region}_{nodo}", cat="Binary")
    for region in regiones
    for nodo in nodos
}


# FUNCIÓN OBJETIVO

modelo += (
    pulp.lpSum(costo_fijo_nodo[n] * y[n] for n in nodos)
    + pulp.lpSum(costo_servicio[r][n] * demanda_region[r] * z[r, n] for r in regiones for n in nodos)
), "Costo_total"


# RESTRICCIONES

# Cada región debe asignarse exactamente a un nodo
for r in regiones:
    modelo += pulp.lpSum(z[r, n] for n in nodos) == 1, f"Asignacion_unica_{r}"

# Una región solo puede asignarse a un nodo abierto
for r in regiones:
    for n in nodos:
        modelo += z[r, n] <= y[n], f"Nodo_abierto_{r}_{n}"

# La demanda asignada a cada nodo no puede superar su capacidad
for n in nodos:
    modelo += pulp.lpSum(demanda_region[r] * z[r, n] for r in regiones) <= capacidad_nodo[n], f"Capacidad_{n}"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

for n in nodos:
    if y[n].varValue == 1:
        print(f"Nodo abierto: {n}")

for r in regiones:
    for n in nodos:
        if z[r, n].varValue == 1:
            print(f"{r} asignada a {n}")

for n in nodos:
    demanda_asignada = sum(demanda_region[r] * z[r, n].varValue for r in regiones)
    print(f"Utilización de {n}: {demanda_asignada}/{capacidad_nodo[n]}")

print("Costo total ($):", pulp.value(modelo.objective))


Estado: Optimal
Nodo abierto: N1
Nodo abierto: N3
R1 asignada a N1
R2 asignada a N1
R3 asignada a N3
R4 asignada a N3
Utilización de N1: 75.0/80
Utilización de N2: 0.0/70
Utilización de N3: 55.0/75
Costo total ($): 525.0



##  Reto de ampliación

Analice las siguientes modificaciones:

1. Exigir que se abran al menos **2 nodos**.
2. Exigir que se abran exactamente **2 nodos**.
3. Imponer que **R1 no pueda utilizar N3** debido a un SLA de latencia.

Compare las soluciones obtenidas.


In [106]:
# RETO EJERCICIO 5
import pulp

nodos = ["N1", "N2", "N3"]
regiones = ["R1", "R2", "R3", "R4"]

capacidad_nodo = {"N1": 80, "N2": 70, "N3": 75}
costo_fijo_nodo = {"N1": 100, "N2": 90, "N3": 95}
demanda_region = {"R1": 40, "R2": 35, "R3": 30, "R4": 25}

costo_servicio = {
    "R1": {"N1": 2, "N2": 5, "N3": 7},
    "R2": {"N1": 4, "N2": 2, "N3": 6},
    "R3": {"N1": 6, "N2": 3, "N3": 2},
    "R4": {"N1": 7, "N2": 5, "N3": 2},
}

nodos_minimos = 2
nodos_exactos = 2

modelo = pulp.LpProblem("Localizacion_Nodos_Edge_Reto", pulp.LpMinimize)

y = {nodo: pulp.LpVariable(f"Abre_{nodo}", cat="Binary") for nodo in nodos}
z = {
    (region, nodo): pulp.LpVariable(f"Asigna_{region}_{nodo}", cat="Binary")
    for region in regiones
    for nodo in nodos
}


# FUNCIÓN OBJETIVO

modelo += (
    pulp.lpSum(costo_fijo_nodo[n] * y[n] for n in nodos)
    + pulp.lpSum(costo_servicio[r][n] * demanda_region[r] * z[r, n] for r in regiones for n in nodos)
), "Costo_total"


# RESTRICCIONES

# Cada región debe asignarse exactamente a un nodo
for r in regiones:
    modelo += pulp.lpSum(z[r, n] for n in nodos) == 1, f"Asignacion_unica_{r}"

# Una región solo puede asignarse a un nodo abierto
for r in regiones:
    for n in nodos:
        modelo += z[r, n] <= y[n], f"Nodo_abierto_{r}_{n}"

# La demanda asignada a cada nodo no puede superar su capacidad
for n in nodos:
    modelo += pulp.lpSum(demanda_region[r] * z[r, n] for r in regiones) <= capacidad_nodo[n], f"Capacidad_{n}"

# Reto 3: R1 no puede utilizar N3 (restricción de SLA)
modelo += z["R1", "N3"] == 0, "SLA_R1_sin_N3"

# Reto 1 y 2: número de nodos abiertos
# Para "al menos 2 nodos" usar >=, para "exactamente 2 nodos" usar ==
modelo += pulp.lpSum(y[n] for n in nodos) >= nodos_minimos, "Minimo_nodos_abiertos"
# modelo += pulp.lpSum(y[n] for n in nodos) == nodos_exactos, "Exactos_nodos_abiertos"


# RESOLVER MODELO

modelo.solve()


# RESULTADOS

print("Estado:", pulp.LpStatus[modelo.status])

for n in nodos:
    if y[n].varValue == 1:
        print(f"Nodo abierto: {n}")

for r in regiones:
    for n in nodos:
        if z[r, n].varValue == 1:
            print(f"{r} asignada a {n}")

for n in nodos:
    demanda_asignada = sum(demanda_region[r] * z[r, n].varValue for r in regiones)
    print(f"Utilización de {n}: {demanda_asignada}/{capacidad_nodo[n]}")

print("Costo total ($):", pulp.value(modelo.objective))

Estado: Optimal
Nodo abierto: N1
Nodo abierto: N3
R1 asignada a N1
R2 asignada a N1
R3 asignada a N3
R4 asignada a N3
Utilización de N1: 75.0/80
Utilización de N2: 0.0/70
Utilización de N3: 55.0/75
Costo total ($): 525.0



#  Ejercicio 6 — Dimensionamiento de agentes de CI/CD

## Planteamiento

Una plataforma DevOps necesita capacidad concurrente para pipelines Linux y Windows.

Existen tres tipos de agentes con diferentes capacidades y costos.

###  Datos

| Tipo de agente | Linux slots | Windows slots | Costo |
|---|---:|---:|---:|
| Standard | 4 | 2 | 50 |
| Linux Optimized | 8 | 0 | 70 |
| Universal | 3 | 5 | 80 |

### Condiciones

- Se requieren al menos **40 slots Linux**.
- Se requieren al menos **20 slots Windows**.
- Deben existir al menos **2 agentes Universal**.
- El equipo de operaciones puede administrar como máximo **12 agentes**.

---

##  Trabajo del estudiante

Formule y resuelva un modelo de programación entera que minimice el costo total.

Debe determinar:

- cantidad de agentes Standard;
- cantidad de agentes Linux Optimized;
- cantidad de agentes Universal;
- costo mínimo;
- slots Linux obtenidos;
- slots Windows obtenidos;
- total de agentes utilizados.


In [109]:
# EJERCICIO 6
# Escriba aquí su modelo en PuLP.
import pulp

modelo = pulp.LpProblem("Agentes_CICD", pulp.LpMinimize)

S = pulp.LpVariable("Standard", lowBound=0, cat="Integer")
L = pulp.LpVariable("Linux_Optimized", lowBound=0, cat="Integer")
U = pulp.LpVariable("Universal", lowBound=0, cat="Integer")

# Funcion objetivo
modelo += 50*S + 70*L + 80*U

# Requerimiento Linux
modelo += 4*S + 8*L + 3*U >= 40, "Req_Linux"

# Requerimiento Windows
modelo += 2*S + 5*U >= 20, "Req_Windows"

# Minimo de agentes Universal
modelo += U >= 2, "Min_Universal"

# Maximo de agentes administrables
modelo += S + L + U <= 12, "Max_agentes"

modelo.solve()

print("Estado:", pulp.LpStatus[modelo.status])
print("Standard =", S.varValue)
print("Linux Optimized =", L.varValue)
print("Universal =", U.varValue)
print("Costo mínimo =", pulp.value(modelo.objective))
print("Slots Linux =", 4*S.varValue + 8*L.varValue + 3*U.varValue)
print("Slots Windows =", 2*S.varValue + 5*U.varValue)
print("Total agentes =", S.varValue + L.varValue + U.varValue)

Estado: Optimal
Standard = 5.0
Linux Optimized = 2.0
Universal = 2.0
Costo mínimo = 550.0
Slots Linux = 42.0
Slots Windows = 20.0
Total agentes = 9.0



## Reto de ampliación

Modifique el modelo de la siguiente manera:

1. Aumente el requerimiento de Windows a **30 slots**.
2. Agregue la regla:

> Por cada 3 agentes Linux Optimized debe existir al menos 1 agente Universal.

Formule matemáticamente dicha restricción e incorpórela al modelo.

Compare el nuevo costo con el problema original.


In [111]:
# RETO EJERCICIO 6
import pulp

modelo_reto = pulp.LpProblem("Agentes_CICD_reto", pulp.LpMinimize)

S = pulp.LpVariable("Standard", lowBound=0, cat="Integer")
L = pulp.LpVariable("Linux_Optimized", lowBound=0, cat="Integer")
U = pulp.LpVariable("Universal", lowBound=0, cat="Integer")

# Funcion objetivo
modelo_reto += 50*S + 70*L + 80*U

# Requerimiento Linux
modelo_reto += 4*S + 8*L + 3*U >= 40, "Req_Linux"

# Requerimiento Windows (aumentado a 30)
modelo_reto += 2*S + 5*U >= 30, "Req_Windows"

# Minimo de agentes Universal
modelo_reto += U >= 2, "Min_Universal"

# Maximo de agentes administrables
modelo_reto += S + L + U <= 12, "Max_agentes"

# Regla nueva: por cada 3 Linux Optimized, al menos 1 Universal
modelo_reto += L <= 3*U, "Regla_LinuxOpt_Universal"

modelo_reto.solve()

print("RETO - Ejercicio 6")
print("Standard =", S.varValue)
print("Linux Optimized =", L.varValue)
print("Universal =", U.varValue)
print("Costo mínimo =", pulp.value(modelo_reto.objective))
print("Slots Linux =", 4*S.varValue + 8*L.varValue + 3*U.varValue)
print("Slots Windows =", 2*S.varValue + 5*U.varValue)
print("Total agentes =", S.varValue + L.varValue + U.varValue)

RETO - Ejercicio 6
Standard = 8.0
Linux Optimized = 0.0
Universal = 3.0
Costo mínimo = 640.0
Slots Linux = 41.0
Slots Windows = 31.0
Total agentes = 11.0



#  Entrega sugerida

Para cada ejercicio, entregue:

- formulación matemática;
- código en PuLP;
- estado del solver;
- valores de las variables;
- valor de la función objetivo;
- comprobación de restricciones;
- interpretación breve de la solución.

> Si el estado del modelo no es `Optimal`, no interprete los valores de las variables como una solución óptima.
